# Lab 2 · A 1D CNN on Real Clinical Spectra
**MSACL · DS301 Deep Learning · Segment 2 · Lab 2**

Last lab you built a network that reads a *table* of metabolite numbers. This lab, you'll
build one that reads a **spectrum** — the same 1D-CNN idea from Lecture 5, trained on real
**MALDI-TOF mass spectra** to call antimicrobial resistance straight from the instrument's
raw signal.

This is a ~45-minute session (+ 10 min discussion). **You fill in five short pieces**, all
about the new parts: the training loop reused on spectra, the CNN itself, and the batch size.
Everything else runs as-is.


## About the data: DRIAMS MALDI-TOF spectra

**DRIAMS** is a public database of routine clinical **MALDI-TOF mass spectra** paired with
antimicrobial-resistance lab results — the same kind of spectrum a clinical microbiology lab
already collects to identify a pathogen, here reused to predict whether it will respond to an
antibiotic *before* the days-long culture-based test comes back.

Today's task: **_Staphylococcus aureus_ + oxacillin** — the classic **MRSA** call. Each sample
is one spectrum, already reduced by the instrument's own processing pipeline to a fixed
**6,000-number vector** (intensity at each of 6,000 mass positions), with a label:

- **0 = susceptible** (oxacillin will likely work)
- **1 = resistant** (MRSA — a different antibiotic is needed)

> **Note:** this lab runs on a smaller *development slice* of DRIAMS (one hospital site,
> 738 spectra) — the full course dataset is larger and better balanced. The code here works
> identically either way; only the data file changes.


## ⚙️ Setup — run this cell first

Run the next cell **before anything else**. On Colab it installs the course's
packages and downloads the data for this lab, checking every file's fingerprint;
on your own computer it uses the files already in the course folder. It takes
about a minute the first time. You do not need to read or edit it.

In [ ]:
# ── MSACL DS301 · SETUP — run this cell first; no need to read or edit it ──
import sys, subprocess, urllib.request, pathlib
if "google.colab" in sys.modules:          # on Colab: fetch the course helper
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "uv"], check=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/indigobio/msacl_dl_course/main/student_pack/msacl.py",
        "msacl.py")
else:                                        # locally: use the course folder's copy
    here = pathlib.Path.cwd()
    helper = next(p for d in (here, *here.parents)
                  for p in (d / "msacl.py", d / "student_pack" / "msacl.py") if p.exists())
    sys.path.insert(0, str(helper.parent))
import msacl
DATA = msacl.setup("lab02")                  # {file name: where it is on this machine}

In [ ]:
# Read and run — no need to edit.
import os
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)
print("Libraries ready.")


## Step 1 · Load the spectra *(runs for you)*

In [ ]:
# Read and run — no need to edit.
# The setup cell at the top already fetched this slice and checked its fingerprint.
data = np.load(DATA["driams_c_saureus_oxacillin.npz"])
X = data["X"].astype("float32")   # shape (738, 6000): raw binned intensities
y = data["y"].astype("float32")   # 0 = susceptible, 1 = resistant

print("Spectra:", X.shape, "| resistant:", int(y.sum()), "/", len(y))


Here's what two real spectra look like — one susceptible, one resistant. Visually they're
very similar; the network's job is to find the subtle pattern a human eye can't easily spot.


In [ ]:
# Read and run — no need to edit.
sus_row = np.where(y == 0)[0][0]
res_row = np.where(y == 1)[0][0]

plt.figure(figsize=(8, 3))
plt.plot(X[sus_row], linewidth=0.8, label="susceptible")
plt.plot(X[res_row], linewidth=0.8, label="resistant", alpha=0.8)
plt.xlabel("mass position (bin, 0-5999)"); plt.ylabel("intensity")
plt.title("Two real MALDI-TOF spectra")
plt.legend()
plt.show()


## Step 2 · Split, balance-check, and normalize — by hand *(runs for you)*

Only **41 of 738** spectra are resistant (~5.6%) — a realistic clinical imbalance (MRSA is
the minority case). A plain shuffle-split could easily starve the test set of resistant
examples entirely, so we split **within each class separately**, then combine — this is
called a **stratified split**, and it guarantees both the train and test sets contain
resistant examples.

We also **normalize each spectrum by its own peak intensity** — instrument runs vary in
overall signal strength, so this puts every spectrum on the same 0–1 scale before training.


In [ ]:
# Read and run — no need to edit.
# --- stratified 80/20 split: split resistant and susceptible indices separately ---
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
np.random.shuffle(pos_idx)
np.random.shuffle(neg_idx)

n_pos_train = int(0.8 * len(pos_idx))
n_neg_train = int(0.8 * len(neg_idx))

train_idx = np.concatenate([pos_idx[:n_pos_train], neg_idx[:n_neg_train]])
test_idx = np.concatenate([pos_idx[n_pos_train:], neg_idx[n_neg_train:]])
np.random.shuffle(train_idx)   # mix classes together within the split
np.random.shuffle(test_idx)

# --- normalize: divide each spectrum by its own maximum intensity ---
X_norm = X / X.max(axis=1, keepdims=True)

X_train, y_train = X_norm[train_idx], y[train_idx]
X_test, y_test = X_norm[test_idx], y[test_idx]

print(f"train: {len(train_idx)} spectra ({int(y_train.sum())} resistant)")
print(f"test:  {len(test_idx)} spectra ({int(y_test.sum())} resistant)")

# tensors: add a "channel" dimension of size 1 for Conv1d -> shape (N, 1, 6000)
X_train_t = torch.tensor(X_train).unsqueeze(1)
X_test_t = torch.tensor(X_test).unsqueeze(1)
y_train_t = torch.tensor(y_train).unsqueeze(1)
y_test_t = torch.tensor(y_test).unsqueeze(1)

n_pos, n_neg = int(y_train.sum()), int(len(y_train) - y_train.sum())
print(f"class imbalance: {n_pos} resistant vs {n_neg} susceptible "
      f"({100 * n_pos / len(y_train):.1f}% resistant) — the rare-class problem of Lecture 10")


## Step 3 · The training loop — YOUR TURN ✏️

This is the **exact same recipe** from Lecture 2 / Lab 1 — forward, loss, zero, backward,
step — reused here for spectra, in **mini-batches** (train on a handful of spectra at a
time, many times per epoch, rather than all 590 at once). We wrap it in a function so we
can reuse it for the different comparisons below, with a **live progress bar** showing the
loss as it trains.

**Fill in the five lines** inside the batch loop, in order (same names as Lab 1):


In [ ]:
def train_model(model, X, y, epochs, batch_size, lr, loss_fn):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    n = X.shape[0]
    loss_history = []

    pbar = tqdm(range(epochs), desc="training")
    for epoch in pbar:
        order = torch.randperm(n)                       # shuffle sample order each epoch
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            batch_idx = order[start:start + batch_size]
            X_batch, y_batch = X[batch_idx], y[batch_idx]

            # ========== YOUR CODE HERE ==========
            ...  # replace with your code

            epoch_loss += loss.item() * len(batch_idx)
        avg_loss = epoch_loss / n
        loss_history.append(avg_loss)
        pbar.set_postfix(loss=f"{avg_loss:.3f}")          # live loss monitor on the progress bar
    return loss_history

# --- self-check: do not edit ---
_probe_model = nn.Linear(4, 1)
_probe_X = torch.randn(20, 4)
_probe_y = torch.randint(0, 2, (20, 1)).float()
_probe_history = train_model(_probe_model, _probe_X, _probe_y, epochs=2, batch_size=5,
                              lr=0.01, loss_fn=nn.BCEWithLogitsLoss())
assert len(_probe_history) == 2, f"expected 2 loss values (one per epoch), got {len(_probe_history)}"
assert all(isinstance(v, float) for v in _probe_history), "loss_history should hold plain floats"
print("Loop checks out — 2 epochs ran, loss history:", [round(v, 3) for v in _probe_history])


In [ ]:
# Read and run — no need to edit.
def evaluate_model(model, X, y, threshold=0.5):
    with torch.no_grad():
        probs = torch.sigmoid(model(X))
    predicted = (probs > threshold).float()               # call 'resistant' when prob clears the threshold
    accuracy = (predicted == y).float().mean().item()
    # resistant recall: of the truly-resistant cases, how many did we catch?
    is_resistant = (y == 1)
    if is_resistant.sum() > 0:
        recall = (predicted[is_resistant] == 1).float().mean().item()
    else:
        recall = float("nan")
    return accuracy, recall

print("Evaluation helper ready.")


## Step 4 · Build the 1D CNN — YOUR TURN ✏️

A CNN doesn't need every position wired to every neuron — it slides a small **filter** along
the m/z axis, the same "peak detector everywhere" idea from Lecture 5. Fill in **three** blanks:

**Blank 1 — the first `Conv1d`.** Use exactly the arguments from the Lecture 5 code slide:
one input channel (a raw spectrum), 8 filters, a filter that's 15 bins wide, and padding so
the output stays the same length.

**Blank 2 — a second conv block.** Mirror block 1's pattern (`Conv1d` → `ReLU` → `MaxPool1d`),
but now reading the **8** channels block 1 produced and turning them into **16**.

**Blank 3 — the input size of the first `Linear` layer.** After the conv blocks you have to
flatten the feature maps into one long vector to hand to a fully-connected layer — and you must
tell `nn.Linear` exactly how long that vector is. Work it out from the pieces:

- the **`Conv1d` layers keep the length unchanged** — that's what the `padding` is for, so a
  spectrum goes in 6,000 bins long and stays 6,000 bins long through each convolution;
- each **`MaxPool1d(4)` divides the length by 4** (it keeps the largest value in every window of
  4), and there are **two** of them;
- **`Flatten`** then stacks all **16** channels of that pooled length into one vector.

So the flattened length `fc1` must accept is **16 channels × (the pooled length)**. Compute the
pooled length first, then multiply. If you get it wrong, the self-check below re-computes the true
size from the actual model and tells you what it should have been.


In [ ]:
class SpectrumCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # ========== YOUR CODE HERE ==========
        ...  # replace with your code
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(4)     # halve? no — divide the length by 4

        # ========== YOUR CODE HERE ==========
        ...  # replace with your code

        self.flatten = nn.Flatten()               # stacks the 16 channels into one long vector
        # ========== YOUR CODE HERE ==========
        ...  # replace with your code
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        return self.fc2(x)

# --- self-checks: do not edit ---
_probe = SpectrumCNN()
_out1 = _probe.pool1(_probe.relu1(_probe.conv1(torch.zeros(2, 1, 6000))))
assert _out1.shape[:2] == (2, 8), f"expected 8 channels after block 1, got {tuple(_out1.shape)}"
_out2 = _probe.pool2(_probe.relu2(_probe.conv2(_out1)))
assert _out2.shape[:2] == (2, 16), f"expected 16 channels after block 2, got {tuple(_out2.shape)}"
assert _probe.fc1.in_features == _out2.shape[1] * _out2.shape[2], \
    f"fc1 must accept channels x pooled length = {_out2.shape[1] * _out2.shape[2]}, got {_probe.fc1.in_features}"
assert _probe(torch.zeros(2, 1, 6000)).shape == (2, 1), "a full forward pass should return one logit per spectrum"
print("Shapes check out — fc1 accepts", _probe.fc1.in_features, "features; forward pass returns one logit per spectrum.")


## Step 5 · Choose a batch size — YOUR TURN ✏️

Bigger batches give a smoother, more confident gradient but fewer updates per epoch (and can
generalize *worse*, from Lecture 4); tiny batches are noisy but update often. Pick a
reasonable **middle-ground** batch size for 590 training spectra.


In [ ]:
# ========== YOUR CODE HERE ==========
...  # replace with your code

# --- self-check: do not edit ---
assert 4 <= CNN_BATCH_SIZE <= 128, "pick a batch size between 4 and 128 - see Lecture 4's Goldilocks range"
print("Using batch size:", CNN_BATCH_SIZE)


## Step 6 · Train the CNN *(runs for you)*

Watch the progress bar's loss shrink as it trains.


In [ ]:
# Read and run — no need to edit.
loss_fn = nn.BCEWithLogitsLoss()          # plain binary cross-entropy — every spectrum weighted equally

torch.manual_seed(0)              # fixed seed -> everyone trains the same CNN (so your threshold exploration below is reproducible)
cnn = SpectrumCNN()
cnn_losses = train_model(cnn, X_train_t, y_train_t, epochs=10, batch_size=CNN_BATCH_SIZE,
                          lr=0.001, loss_fn=loss_fn)
cnn_acc, cnn_recall = evaluate_model(cnn, X_test_t, y_test_t)   # uses the default 0.5 threshold

plt.figure(figsize=(6, 4))
plt.plot(cnn_losses)
plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("CNN training loss (plain BCE)")
plt.show()

print(f"CNN @ threshold 0.5  ->  test accuracy: {cnn_acc:.2f}   resistant recall: {cnn_recall:.2f}")


**Read the recall number, not just accuracy.** With ~94% of spectra susceptible, a lazy
model that always guesses "susceptible" already scores ~94% accuracy while catching **zero**
resistant cases — exactly the trap Lecture 10 warns about. And that's very nearly what plain
BCE does here: trained on 17-to-1 imbalanced data, it learns to output a *low* probability for
almost every spectrum, so at the **default 0.5 threshold** it flags hardly any resistant cases
and recall looks terrible.

But don't write the model off yet — that dismal recall is an artifact of *where we drew the
line*, not proof the model learned nothing. The next step shows why.


## Step 7 · Recall depends on the threshold — YOUR TURN ✏️

The model doesn't output "resistant" or "susceptible" — it outputs a **probability**. We call a
spectrum resistant when that probability clears a **threshold**, and `0.5` is just a default,
not a law of nature. On 17-to-1 imbalanced data the model's probabilities pile up *low*, so
`0.5` sits above almost all of them — a terrible place to draw the line, and recall stays near
zero.

**Your job:** explore a few thresholds and find one that catches at least **half** the resistant
cases (**recall ≥ 0.5**). Start at `0.5`, then try lower cuts and watch two things move in
opposite directions — as you drop the threshold, **recall climbs** (you catch more resistant
cases) but **precision and accuracy fall** (you raise more false alarms). There is no universal
"right" answer: it's a **clinical decision** (Lecture 10).


In [ ]:
# YOUR TURN: change `t`, re-run, and read what happens to recall vs accuracy.
# Least-code way: just edit the number and press Shift-Enter again a few times.
# (Comfortable with a loop? You could instead write:  for t in [0.5, 0.1, 0.05]: ... )
t = 0.5
acc, recall = evaluate_model(cnn, X_test_t, y_test_t, threshold=t)
print(f"threshold {t:.2f}  ->  recall {recall:.2f}   accuracy {acc:.2f}")


As you lower `t`, **recall climbs**: the model *did* learn a usable signal — at `0.5` we were
just slicing it in the wrong place. The price is precision and accuracy, since lower cuts flag
more false alarms.

For an MRSA screen you'd tolerate some false alarms rather than miss a resistant case, so you'd
pick the threshold on a validation set to hit a **required recall** — never blindly trust `0.5`.
Now commit to a cut below.


## Step 8 · Commit to a threshold — YOUR TURN ✏️

From your exploration above, set **`chosen_threshold`** to a cut that catches at least half the resistant cases (**recall ≥ 0.5**). The self-check re-evaluates the model at *your* threshold and confirms the recall target is met.


In [ ]:
# YOUR TURN: the threshold you found above that catches at least half the resistant cases.
# ========== YOUR CODE HERE ==========
...  # replace with your code

acc_at_cut, recall_at_cut = evaluate_model(cnn, X_test_t, y_test_t, threshold=chosen_threshold)
print(f"at your threshold {chosen_threshold:.3f}  ->  "
      f"resistant recall {recall_at_cut:.2f}   accuracy {acc_at_cut:.2f}")

# --- self-check: do not edit ---
assert recall_at_cut >= 0.5, "pick a lower threshold - you need to catch at least half the resistant cases"
print("Recall target met: you catch at least half the resistant cases.")


## What just happened

You adapted the Lab 1 recipe to a real clinical signal:
- the **same five-line training loop**, now wrapped for **mini-batches** with a live progress bar,
- a **1D CNN** that slides filters along the m/z axis instead of flattening everything,
- an honest look at **imbalanced classification** — accuracy alone would have hidden a model
  that never catches a single resistant case,
- and the key idea that **recall depends on the decision threshold**: 0.5 is an arbitrary default,
  and on rare-class data you choose the cut to hit a clinical target, not the other way around.

With only 738 spectra from one site, don't expect huge numbers — the point is the *shape* of
the recipe, which is identical on the full production dataset.


## Optional stretch: a loss built for rare cases — focal loss *(only if you're ahead)*

Instead of moving the threshold *after* training, can we train a model whose probabilities aren't so
squashed toward zero? **Focal loss** (Lecture 4) does two things: it **down-weights** examples the
model already gets right confidently (so training concentrates on the hard, easy-to-miss cases), and
its `alpha` term **tilts the balance toward the rare resistant class**. The result: more resistant
spectra push their probability up past 0.5 on their own. Train it and compare recall at the *same*
default 0.5 cut.


In [ ]:
# Optional - read and run, or tweak alpha / gamma yourself.
def focal_loss(logits, targets, alpha=0.946, gamma=2.0):
    p = torch.sigmoid(logits)
    ce = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    focal_w = (targets - p).abs().pow(gamma)                 # shrinks toward 0 for easy/confident cases
    alpha_t = targets * alpha + (1 - targets) * (1 - alpha)  # tilts the balance toward the rare resistant class
    return (alpha_t * focal_w * ce).mean()

cnn_focal = SpectrumCNN()
focal_losses = train_model(cnn_focal, X_train_t, y_train_t, epochs=10, batch_size=CNN_BATCH_SIZE,
                            lr=0.001, loss_fn=focal_loss)
focal_acc, focal_recall = evaluate_model(cnn_focal, X_test_t, y_test_t)   # same default 0.5 threshold

print(f"plain BCE   @ 0.5  ->  accuracy: {cnn_acc:.2f}   recall: {cnn_recall:.2f}")
print(f"focal loss  @ 0.5  ->  accuracy: {focal_acc:.2f}   recall: {focal_recall:.2f}")


## Discussion (10 min)

**Where in your workflow is there a 1D signal a CNN could read? Who labels it today, and how
well?** Think of a chromatogram, a QC trace, or another spectrum you see routinely — and how
confident a human currently is when reading it.
